<a href="https://colab.research.google.com/github/Ajeetesh1999/ExpressMySQL/blob/master/Copy_of_pytorch_basics_tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PyTorch from Scratch — Tutorial

> **Goal:** Understand tensors deeply — scalars, vectors, matrices, shape quirks, and how to build your first model.

In [ ]:
# Run this first — imports everything we need
import torch
import torch.nn as nn
import numpy as np

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
print("\n✅ Setup complete — let's go!")

PyTorch version : 2.11.0+cpu
CUDA available  : False

✅ Setup complete — let's go!


---
## Section 1 — Why move from sklearn to PyTorch? 🤔

sklearn is fantastic for classical ML. But it has hard limits:

| Limitation | sklearn | PyTorch |
|---|---|---|
| **GPU acceleration** | ❌ CPU only | ✅ Native CUDA support |
| **Custom architectures** | ❌ Fixed models | ✅ Build anything |
| **Automatic gradients** | ❌ Not available | ✅ Autograd built-in |
| **Large-scale training** | ❌ Data must fit in RAM | ✅ Minibatch streaming |
| **Deep learning** | ⚠️ Very limited | ✅ First-class support |

**Rule of thumb:**
- 📊 Tabular data, quick baselines → use **sklearn**
- 🧠 Deep learning, custom models, GPU needed → use **PyTorch**

In [ ]:
# A taste of the difference — same linear regression, two frameworks

# --- sklearn style ---
from sklearn.linear_model import LinearRegression
import numpy as np

X_np = np.random.rand(100, 4)
y_np = np.random.rand(100, 1)

sk_model = LinearRegression()
sk_model.fit(X_np, y_np)
print("sklearn coef shape:", sk_model.coef_.shape)  # (1, 4)

# --- PyTorch style ---
X_pt = torch.tensor(X_np, dtype=torch.float32)
y_pt = torch.tensor(y_np, dtype=torch.float32)

pt_model = nn.Linear(4, 1)
print("PyTorch weight shape:", pt_model.weight.shape)  # torch.Size([1, 4])

print("\n✅ Same concept, different power — PyTorch gives you the training loop.")

---
## Section 2 — Scalar (0 dimensions) ⚫

A **scalar** is a single number. In PyTorch it's a tensor with **no dimensions at all**.

```
Shape: torch.Size([])   ← empty brackets = 0 dimensions
ndim:  0
```

> **When you'll see it:** Loss values are scalars. `loss.item()` extracts the Python float from a scalar tensor.

In [ ]:
# ── Creating scalars ──────────────────────────────────────────────────
x = torch.tensor(42.0)

print("Value  :", x)           # tensor(42.)
print("Shape  :", x.shape)     # torch.Size([])
print("ndim   :", x.ndim)      # 0
print("dtype  :", x.dtype)     # torch.float32
print("item() :", x.item())    # 42.0  ← Python float

print("\n--- Scalar arithmetic ---")
a = torch.tensor(3.0)
b = torch.tensor(4.0)
print("a + b  :", a + b)       # tensor(7.)
print("a * b  :", a * b)       # tensor(12.)
print("a ** 2 :", a ** 2)      # tensor(9.)

In [ ]:
# ── Real use-case: loss is always a scalar ────────────────────────────
pred = torch.tensor([2.5, 0.0, 2.0, 8.0])
true = torch.tensor([3.0, -0.5, 2.0, 7.0])

loss = torch.mean((pred - true) ** 2)   # MSE loss
print("Loss tensor :", loss)
print("Loss shape  :", loss.shape)      # torch.Size([])  ← scalar!
print("Loss value  :", loss.item())     # Python float for logging

---
## Section 3 — Vector (1 dimension) ➡️

A **vector** is a 1D list of numbers. Think of it as a sequence — no concept of "row" or "column" yet.

```
[ 1.0,  2.0,  3.0,  4.0,  5.0 ]
   ↑                         ↑
 index 0                  index 4

Shape: torch.Size([5])   ← one number = one dimension
ndim:  1
```

In [ ]:
# ── Creating vectors ──────────────────────────────────────────────────
v = torch.tensor([1.0, 2.0, 3.0, 4.0, 5.0])

print("Vector :", v)
print("Shape  :", v.shape)    # torch.Size([5])
print("ndim   :", v.ndim)     # 1
print("len    :", len(v))     # 5

print("\n--- Indexing & slicing ---")
print("v[0]   :", v[0])       # first element
print("v[-1]  :", v[-1])      # last element
print("v[1:3] :", v[1:3])     # slice

print("\n--- Useful constructors ---")
print("zeros  :", torch.zeros(5))
print("ones   :", torch.ones(5))
print("arange :", torch.arange(0, 5, 1))    # like np.arange
print("linspace:", torch.linspace(0, 1, 5)) # 5 evenly spaced values

In [ ]:
# ── Vector operations ─────────────────────────────────────────────────
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([4.0, 5.0, 6.0])

print("a + b          :", a + b)             # element-wise add
print("a * b          :", a * b)             # element-wise multiply
print("dot product    :", torch.dot(a, b))   # 1*4 + 2*5 + 3*6 = 32
print("a.sum()        :", a.sum())           # scalar
print("a.mean()       :", a.mean())          # scalar
print("L2 norm        :", torch.norm(a))     # sqrt(1+4+9)

---
## Section 4 — Matrix (2 dimensions) 🔲

A **matrix** has rows and columns. Shape is always `[rows, cols]`.

```
         col 0  col 1  col 2  col 3
row 0  [  1,     2,     3,     4  ]
row 1  [  5,     6,     7,     8  ]
row 2  [  9,    10,    11,    12  ]

Shape: torch.Size([3, 4])   ← first dim = rows, second = cols
ndim:  2
```

> **This is your dataset!** 100 samples × 4 features = shape `[100, 4]`

In [ ]:
# ── Creating matrices ─────────────────────────────────────────────────
M = torch.tensor([
    [1,  2,  3,  4],
    [5,  6,  7,  8],
    [9, 10, 11, 12]
], dtype=torch.float32)

print("Matrix M:\n", M)
print("\nShape :", M.shape)    # torch.Size([3, 4])
print("ndim  :", M.ndim)      # 2
print("numel :", M.numel())   # 12 total elements

print("\n--- Indexing ---")
print("M[1, 2]  :", M[1, 2])   # row 1, col 2  → 7
print("M[0, :]  :", M[0, :])   # entire row 0
print("M[:, 1]  :", M[:, 1])   # entire col 1

In [ ]:
# ── Matrix operations ─────────────────────────────────────────────────
A = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
B = torch.tensor([[5.0, 6.0], [7.0, 8.0]])

print("A + B (element-wise):\n", A + B)
print("\nA * B (element-wise):\n", A * B)   # NOT matrix multiply!
print("\nA @ B (matrix multiply):\n", A @ B)  # the real matmul

print("\n--- Reductions ---")
print("sum all       :", A.sum())
print("sum over rows :", A.sum(dim=0))  # collapse rows → shape [2]
print("sum over cols :", A.sum(dim=1))  # collapse cols → shape [2]

print("\n--- Transpose ---")
print("A.T shape:", A.T.shape)  # (2,2) → (2,2), but for non-square:
X = torch.rand(3, 5)
print("X.T shape:", X.T.shape)  # (3,5) → (5,3)

---
## Section 5 — Tensor (N dimensions) 🧊

A **tensor** generalises everything. Scalar, vector, matrix — they are all tensors.

| Name | Rank | Shape example | Real-world meaning |
|---|---|---|---|
| Scalar | 0 | `[]` | A single loss value |
| Vector | 1 | `[512]` | Embedding of a word |
| Matrix | 2 | `[100, 4]` | 100 samples, 4 features |
| 3D Tensor | 3 | `[32, 128, 512]` | Batch of sequences |
| 4D Tensor | 4 | `[32, 3, 64, 64]` | Batch of RGB images |

In [ ]:
# ── 3D tensor — a batch of matrices ──────────────────────────────────
# Think of it as: 4 matrices, each of size (3, 5)
T = torch.zeros(4, 3, 5)
print("T.shape :", T.shape)    # torch.Size([4, 3, 5])
print("T.ndim  :", T.ndim)     # 3
print("T.numel :", T.numel())  # 60

print("\n--- Accessing slices ---")
print("T[0].shape   :", T[0].shape)     # first 'matrix' → [3, 5]
print("T[0,1].shape :", T[0,1].shape)   # first matrix, row 1 → [5]
print("T[0,1,2]     :", T[0,1,2])       # single element → scalar

In [ ]:
# ── 4D tensor — what images look like ────────────────────────────────
# [batch, channels, height, width]  ← PyTorch convention (NCHW)

# A batch of 32 RGB images at 64×64 resolution:
images = torch.rand(32, 3, 64, 64)

print("images.shape :", images.shape)
print()
print("  dim 0 = batch size      :", images.shape[0], "images")
print("  dim 1 = channels (RGB)  :", images.shape[1], "channels")
print("  dim 2 = height          :", images.shape[2], "pixels")
print("  dim 3 = width           :", images.shape[3], "pixels")

print()
# Get a single image and a single pixel
one_image = images[0]           # shape [3, 64, 64]
one_channel = images[0, 0]      # shape [64, 64]
one_pixel = images[0, 0, 10, 10] # scalar

print("One image shape    :", one_image.shape)
print("One channel shape  :", one_channel.shape)
print("One pixel value    :", one_pixel.item())

---
## Section 6 — The Shape Confusion: `(5,)` vs `(5,1)` vs `(1,5)` ⚠️

This is the #1 source of bugs for people coming from sklearn or NumPy.

```
(5,)   → 1D vector       [a, b, c, d, e]
         rank 1, no row/column concept

(5,1)  → 2D column vector [[a],
          rank 2             [b],
                             [c],
                             [d],
                             [e]]

(1,5)  → 2D row vector   [[a, b, c, d, e]]
          rank 2
```

They all hold the same 5 numbers — but they behave very differently in matrix operations and broadcasting!

In [ ]:
# ── Creating all three forms ──────────────────────────────────────────
a = torch.tensor([1.0, 2.0, 3.0, 4.0, 5.0])  # shape (5,)

b = a.unsqueeze(1)   # (5,) → (5, 1)  ← add a dim at position 1
c = a.unsqueeze(0)   # (5,) → (1, 5)  ← add a dim at position 0

# Or equivalently:
b2 = a.view(5, 1)
c2 = a.view(1, 5)

print("a shape :", a.shape)   # torch.Size([5])
print("b shape :", b.shape)   # torch.Size([5, 1])
print("c shape :", c.shape)   # torch.Size([1, 5])

print("\n--- ndim difference ---")
print("a.ndim :", a.ndim)     # 1
print("b.ndim :", b.ndim)     # 2
print("c.ndim :", c.ndim)     # 2

print("\n--- Going back to 1D ---")
print("b.squeeze().shape :", b.squeeze().shape)  # (5, 1) → (5,)

In [ ]:
# ── Why this matters: matrix multiplication ───────────────────────────
a = torch.tensor([1.0, 2.0, 3.0, 4.0, 5.0])  # (5,)
b = a.unsqueeze(1)  # (5, 1)
c = a.unsqueeze(0)  # (1, 5)

# Outer product: (5,1) @ (1,5) = (5,5)
outer = b @ c
print("Outer product shape:", outer.shape)  # torch.Size([5, 5])

# Dot product: (1,5) @ (5,1) = (1,1)
dot = c @ b
print("Dot product shape  :", dot.shape)   # torch.Size([1, 1])
print("Dot product value  :", dot)

# (5,) @ (5,) = scalar (special case PyTorch handles)
dot2 = torch.dot(a, a)
print("torch.dot shape    :", dot2.shape)  # torch.Size([])

In [ ]:
# ── The sklearn gotcha ────────────────────────────────────────────────
# sklearn returns (N,) labels — PyTorch loss functions often expect (N,) OR (N,1)
# Mismatch causes WRONG silent results!

from sklearn.datasets import make_regression
X_sk, y_sk = make_regression(n_samples=10, n_features=4, noise=0.1)

print("sklearn y shape  :", y_sk.shape)   # (10,)  ← 1D!

y_pt = torch.tensor(y_sk, dtype=torch.float32)
print("After conversion :", y_pt.shape)   # torch.Size([10])

# Fix for a regression model that outputs (10, 1):
y_col = y_pt.unsqueeze(1)
print("As column vector :", y_col.shape)  # torch.Size([10, 1])

print("\n💡 Always check .shape when converting from sklearn/numpy!")

---
## Section 7 — Common Shapes Cheatsheet 📋

Once you recognise these patterns, shapes become second nature:

| Shape | Meaning | Example |
|---|---|---|
| `[N]` | 1D vector | Labels, bias terms |
| `[N, F]` | Feature matrix | N samples, F features |
| `[N, 1]` | Column vector | Regression outputs |
| `[N, C]` | Class logits | Classifier output, C classes |
| `[B, T, D]` | Batch of sequences | NLP, RNN, Transformer input |
| `[B, C, H, W]` | Image batch | CNN input (NCHW format) |

In [ ]:
# ── Reshaping: same data, new view ───────────────────────────────────
x = torch.arange(24, dtype=torch.float32)   # shape [24]
print("Original  :", x.shape)               # [24]

print("view(6,4) :", x.view(6, 4).shape)    # [6, 4]
print("view(2,3,4):", x.view(2,3,4).shape)  # [2, 3, 4]
print("view(-1,4):", x.view(-1, 4).shape)   # [6, 4]  (-1 = infer)
print("view(-1,6):", x.view(-1, 6).shape)   # [4, 6]  (-1 = infer)

print()
# reshape vs view: reshape works on non-contiguous tensors too
# view requires contiguous memory — use reshape when unsure
print("reshape(-1,3):", x.reshape(-1, 3).shape)  # safer than view

In [ ]:
# ── squeeze and unsqueeze — adding/removing size-1 dimensions ────────
x = torch.rand(3, 1, 5, 1)
print("Original           :", x.shape)               # [3,1,5,1]
print("squeeze()          :", x.squeeze().shape)      # [3,5]  removes ALL size-1 dims
print("squeeze(1)         :", x.squeeze(1).shape)     # [3,5,1] removes dim 1 only

y = torch.rand(3, 5)
print("\nOriginal           :", y.shape)               # [3,5]
print("unsqueeze(0)       :", y.unsqueeze(0).shape)   # [1,3,5]
print("unsqueeze(1)       :", y.unsqueeze(1).shape)   # [3,1,5]
print("unsqueeze(-1)      :", y.unsqueeze(-1).shape)  # [3,5,1]

In [ ]:
# ── Broadcasting — NumPy rules apply ─────────────────────────────────
# PyTorch aligns shapes from the RIGHT and expands size-1 dims

A = torch.ones(3, 5)      # shape [3, 5]
b = torch.ones(5)         # shape [5]  → broadcast to [3, 5]
print("(3,5) + (5,)  =>", (A + b).shape)    # [3, 5]  ✅

c = torch.ones(3, 1)      # shape [3, 1]
print("(3,5) + (3,1) =>", (A + c).shape)    # [3, 5]  ✅ expands col

# Intentional error example (uncomment to see):
# d = torch.ones(4)
# A + d   # ❌ RuntimeError: sizes don't match (5 vs 4)

print("\n💡 Broadcasting rule: align from right, each dim must be equal OR one of them must be 1")

---
## Section 8 — dtype and device 🖥️

Every tensor has three attributes that fully describe it:

| Attribute | What it means | Example |
|---|---|---|
| `.shape` | Size of each dimension | `torch.Size([32, 4])` |
| `.dtype` | Number type | `torch.float32` |
| `.device` | Where it lives | `cpu` or `cuda:0` |

In [ ]:
# ── dtype: what kind of numbers ───────────────────────────────────────
a = torch.tensor([1.0, 2.0, 3.0])           # float32 (default)
b = torch.tensor([1, 2, 3])                  # int64 (default)
c = torch.tensor([True, False, True])        # bool

print("float tensor dtype :", a.dtype)       # torch.float32
print("int tensor dtype   :", b.dtype)       # torch.int64
print("bool tensor dtype  :", c.dtype)       # torch.bool

print("\n--- Explicit dtype ---")
x = torch.zeros(3, dtype=torch.float16)     # half precision
y = torch.zeros(3, dtype=torch.int32)
print("float16 :", x.dtype)
print("int32   :", y.dtype)

print("\n--- Casting ---")
z = a.to(torch.float64)
print("a as float64 :", z.dtype)
z2 = a.int()          # shorthand for .to(torch.int32)
print("a as int     :", z2.dtype)

print("\n💡 Model weights use float32. Labels/indices use int64.")

In [ ]:
# ── device: CPU or GPU ────────────────────────────────────────────────
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# Create directly on device
x = torch.tensor([1.0, 2.0, 3.0]).to(device)
print("x.device :", x.device)

# Or create on device from the start
y = torch.zeros(3, 4, device=device)
print("y.device :", y.device)

print()
print("--- Key rules ---")
print("1. All tensors in an operation must be on the SAME device")
print("2. Model AND data must both be moved to GPU")
print("3. Use .cpu() to move back for plotting/numpy conversion")

# Move back to CPU for numpy
x_cpu = x.cpu().numpy()
print("\nConverted to numpy:", x_cpu, type(x_cpu))

In [ ]:
# ── requires_grad: the magic behind training ──────────────────────────
# When True, PyTorch tracks every operation on this tensor
# so it can compute gradients automatically.

w = torch.tensor(3.0, requires_grad=True)   # a "weight"
b = torch.tensor(1.0, requires_grad=True)   # a "bias"

x = torch.tensor(2.0)   # input (no grad needed)

# Forward pass: y = w*x + b
y = w * x + b
print("y                   :", y)        # tensor(7., grad_fn=...)

# Loss: L = (y - target)^2
target = torch.tensor(10.0)
loss = (y - target) ** 2
print("loss                :", loss)

# Backward pass: compute dL/dw and dL/db
loss.backward()

print("\ndL/dw (w.grad)      :", w.grad)  # -12.0
print("dL/db (b.grad)      :", b.grad)  # -6.0
print("\n💡 These gradients are exactly what an optimizer uses to update weights.")

---
## Section 9 — Putting it all together: sklearn → PyTorch 🚀

Let's build the same regression model in sklearn and PyTorch side by side, then extend it to show PyTorch's advantages.

In [ ]:
# ── Generate synthetic regression data ───────────────────────────────
from sklearn.datasets import make_regression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Make data
X_np, y_np = make_regression(n_samples=500, n_features=8,
                               noise=20, random_state=42)
y_np = y_np.reshape(-1, 1)   # (500,) → (500, 1)

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X_np, y_np, test_size=0.2, random_state=42
)

# Scale features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print(f"Train: X={X_train.shape}, y={y_train.shape}")
print(f"Test:  X={X_test.shape},  y={y_test.shape}")

In [ ]:
# ── sklearn baseline ──────────────────────────────────────────────────
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

sk_model = LinearRegression()
sk_model.fit(X_train, y_train)

sk_pred = sk_model.predict(X_test)
sk_mse  = mean_squared_error(y_test, sk_pred)

print(f"sklearn Test MSE: {sk_mse:.2f}")

In [ ]:
# ── PyTorch equivalent ────────────────────────────────────────────────

# 1. Convert numpy → PyTorch tensors
X_tr = torch.tensor(X_train, dtype=torch.float32)
y_tr = torch.tensor(y_train, dtype=torch.float32)  # shape [400, 1]
X_te = torch.tensor(X_test,  dtype=torch.float32)
y_te = torch.tensor(y_test,  dtype=torch.float32)

print("Tensor shapes:")
print(f"  X_tr: {X_tr.shape}   y_tr: {y_tr.shape}")
print(f"  X_te: {X_te.shape}    y_te: {y_te.shape}")

# 2. Define model
model = nn.Linear(in_features=8, out_features=1)

# 3. Loss + Optimizer
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

print(f"\nModel: {model}")
print(f"Parameters: weight={model.weight.shape}, bias={model.bias.shape}")

In [ ]:
# ── The training loop ─────────────────────────────────────────────────
# This pattern is at the heart of EVERY PyTorch model:
#   forward → loss → zero_grad → backward → step

loss_history = []
EPOCHS = 200

for epoch in range(EPOCHS):

    # 1. Forward pass
    y_pred = model(X_tr)                # shape [400, 1]

    # 2. Compute loss (scalar)
    loss = loss_fn(y_pred, y_tr)

    # 3. Zero gradients from previous step
    optimizer.zero_grad()

    # 4. Backward pass — computes all gradients
    loss.backward()

    # 5. Update weights
    optimizer.step()

    loss_history.append(loss.item())

    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1:3d}/{EPOCHS}  |  Loss: {loss.item():.2f}")

print("\n✅ Training complete!")

In [ ]:
# ── Evaluate & compare ────────────────────────────────────────────────
import matplotlib.pyplot as plt

# PyTorch evaluation — disable gradient tracking
model.eval()
with torch.no_grad():
    y_pred_te = model(X_te)
    pt_mse = loss_fn(y_pred_te, y_te).item()

print(f"sklearn  Test MSE : {sk_mse:.2f}")
print(f"PyTorch  Test MSE : {pt_mse:.2f}")
print("(Should be similar — same linear model, same data)")

# Plot training loss
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(loss_history, color='steelblue', linewidth=1.5)
plt.title('Training Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.grid(True, alpha=0.3)

# Predicted vs actual
plt.subplot(1, 2, 2)
plt.scatter(y_te.numpy(), y_pred_te.numpy(), alpha=0.4, color='steelblue', s=20)
lim = [y_te.min().item(), y_te.max().item()]
plt.plot(lim, lim, 'r--', linewidth=1.5, label='Perfect prediction')
plt.title('Predicted vs Actual')
plt.xlabel('True values')
plt.ylabel('Predicted values')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ── Bonus: go deeper — something sklearn simply can't do ──────────────
# Multi-layer neural network in ~10 lines

deep_model = nn.Sequential(
    nn.Linear(8, 64),
    nn.ReLU(),
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Linear(32, 1)
)

optimizer = torch.optim.Adam(deep_model.parameters(), lr=0.001)
loss_history_deep = []

deep_model.train()
for epoch in range(300):
    y_pred = deep_model(X_tr)
    loss = loss_fn(y_pred, y_tr)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    loss_history_deep.append(loss.item())

deep_model.eval()
with torch.no_grad():
    deep_mse = loss_fn(deep_model(X_te), y_te).item()

print(f"sklearn linear   MSE : {sk_mse:.2f}")
print(f"PyTorch linear   MSE : {pt_mse:.2f}")
print(f"PyTorch 3-layer  MSE : {deep_mse:.2f}")
print("\n🎉 More layers = more expressiveness!")

---
## 🎓 Summary — What you learned

| Concept | Key takeaway |
|---|---|
| **Scalar** | 0-dim tensor, shape `[]`. Loss values are scalars. |
| **Vector** | 1-dim tensor, shape `[N]`. A flat sequence. |
| **Matrix** | 2-dim tensor, shape `[R, C]`. Your dataset. |
| **Tensor** | N-dim array. Images `[B,C,H,W]`, sequences `[B,T,D]`. |
| **`(5,)` vs `(5,1)`** | 1D vs 2D — different broadcast and matmul behaviour. |
| **`unsqueeze/squeeze`** | Add/remove size-1 dimensions. |
| **`view/reshape`** | Reinterpret memory with a new shape. |
| **dtype** | `float32` for weights, `int64` for indices/labels. |
| **device** | `.to(device)` moves tensors to CPU/GPU. |
| **Training loop** | forward → loss → zero_grad → backward → step |

---

### 🚀 Next steps
- **Datasets & DataLoaders** — batch your data properly
- **`nn.Module`** — build custom model classes
- **Autograd in depth** — how `backward()` really works
- **CNNs** — `nn.Conv2d` for image tasks
- **Transfer learning** — use pretrained models with `torchvision`